In [5]:
# 3.1 Import Library dan Konfigurasi Path
import pandas as pd
import numpy as np
import os
import nltk
import warnings
from nltk.sentiment.vader import SentimentIntensityAnalyzer

warnings.filterwarnings('ignore')

# Konfigurasi path
DATA_PATH = '../../../stemming/outputs/VTB/data_translated.csv'
OUTPUT_DIR = '../../../stemming/outputs/VTB'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Memastikan Lexicon VADER tersedia
try:
    nltk.data.find('sentiment/vader_lexicon')
except LookupError:
    nltk.download('vader_lexicon')

print("[INFO] Library dan Lexicon VADER berhasil dimuat.")

[INFO] Library dan Lexicon VADER berhasil dimuat.


[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\ASUS\AppData\Roaming\nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


In [6]:
# 3.2 Load Data da Mengsi Data Kosong 
df = pd.read_csv(DATA_PATH)
df_clean = df.copy()

# 1. Isi NaN dengan string kosong
df_clean['teks_translated'] = df_clean['teks_translated'].fillna('')

# 2. Identifikasi baris yang kosong setelah di-strip
mask_kosong = df_clean['teks_translated'].str.strip() == ''

# 3. Isi dengan placeholder netral
df_clean.loc[mask_kosong, 'teks_translated'] = 'neutral'

print(f"Total data setelah pengisian: {len(df_clean)} tweet (konsisten dengan pendekatan lain)")
print(f"Jumlah baris yang diisi placeholder: {mask_kosong.sum()}")

Total data setelah pengisian: 13192 tweet (konsisten dengan pendekatan lain)
Jumlah baris yang diisi placeholder: 0


In [7]:
# 3.3 Inisialisasi Analyzer dan Update Leksikon Politik
# Inisialisasi awal
analyzer = SentimentIntensityAnalyzer()

# LOAD LEKSIKON POLITIK INGGRIS
POLITIK_EN_PATH = '../../../kamus/inset_vader_political_modified_english.csv'
df_politik_en = pd.read_csv(POLITIK_EN_PATH)
df_politik_en['kata'] = df_politik_en['kata'].astype(str).str.strip().str.lower()

# Ubah ke dictionary
politik_en_dict = dict(zip(df_politik_en['kata'], df_politik_en['mean']))

# UPDATE LEKSIKON VADER
analyzer.lexicon.update(politik_en_dict)
print(f"[INFO] Leksikon VADER berhasil diperbarui dengan {len(politik_en_dict)} kata politik Inggris.")

# FUNGSI SCORING
def get_vader_scores(text):
    """Mengambil dictionary skor lengkap menggunakan analyzer yang sudah diupdate"""
    return analyzer.polarity_scores(text)

print("\n[PROSES] Menghitung skor sentimen menggunakan VADER + Political Lexicon...")

# Terapkan scoring ke seluruh kolom terjemahan
scores_list = df_clean['teks_translated'].apply(get_vader_scores)

# Ekstraksi skor
df_clean['neg'] = scores_list.apply(lambda x: x['neg'])
df_clean['neu'] = scores_list.apply(lambda x: x['neu'])
df_clean['pos'] = scores_list.apply(lambda x: x['pos'])
df_clean['compound_score'] = scores_list.apply(lambda x: x['compound'])

print("[INFO] Perhitungan skor selesai.")

[INFO] Leksikon VADER berhasil diperbarui dengan 49 kata politik Inggris.

[PROSES] Menghitung skor sentimen menggunakan VADER + Political Lexicon...
[INFO] Perhitungan skor selesai.


In [8]:
# 3.4 Klasifikasi Sentimen berdasarkan Threshold VADER
def classify_vader(compound):
    if compound >= 0.05:
        return 'positive'
    elif compound <= -0.05:
        return 'negative'
    else:
        return 'neutral'

print("\n[PROSES] Klasifikasi sentimen berdasarkan threshold standar...")
df_clean['sentiment'] = df_clean['compound_score'].apply(classify_vader)

# Tampilkan Distribusi
distribusi = df_clean['sentiment'].value_counts()
persentase = df_clean['sentiment'].value_counts(normalize=True) * 100

print("\n=== DISTRIBUSI SENTIMEN VTB ===")
for label, count in distribusi.items():
    print(f"{label.upper():<10}: {count:>5} tweet ({persentase[label]:.2f}%)")


[PROSES] Klasifikasi sentimen berdasarkan threshold standar...

=== DISTRIBUSI SENTIMEN VTB ===
POSITIVE  :  5421 tweet (41.09%)
NEGATIVE  :  5134 tweet (38.92%)
NEUTRAL   :  2637 tweet (19.99%)


In [9]:
# 3.5 Preview Hasil
print("\n[PREVIEW] 5 Data Pertama:")
preview_cols = ['no', 'teks_processed', 'teks_translated', 'compound_score', 'sentiment']
print(df_clean[preview_cols].head())


[PREVIEW] 5 Data Pertama:
   no                                     teks_processed  \
0   1  ADIL loh untuk yang punya kebijakan publik neg...   
1   2  tertib media online DPR pemerintah jangan spor...   
2   3  harus evaluasi lagi kebijakan bebas visa utama...   
3   4  jangan ngambang pengaturan logis apa undang un...   
4   5  bebas suara dapat memang jamin UU tetapi bebas...   

                                     teks_translated  compound_score sentiment  
0  FAIR, for those who have state public policy, ...          0.7443  positive  
1  The DPR government's orderly online media shou...         -0.2500  negative  
2  You have to re-evaluate the main visa-free pol...         -0.5399  negative  
3    Don't wonder about logical arrangements or laws          0.0000   neutral  
4  Free speech can indeed guarantee the law, but ...          0.5580  positive  


In [10]:
# 3.6 Simpan Hasil Analisis
# Pilih kolom yang relevan untuk file final
final_cols = [
    'no', 'timestamp', 'teks', 'teks_processed', 
    'teks_translated', 'neg', 'neu', 'pos', 
    'compound_score', 'sentiment'
]

OUTPUT_FILE = os.path.join(OUTPUT_DIR, 'sentiment_clean.csv')
df_clean[final_cols].to_csv(OUTPUT_FILE, index=False, encoding='utf-8')

print(f"\n[OUTPUT] Hasil analisis sentimen VTB disimpan di: {OUTPUT_FILE}")


[OUTPUT] Hasil analisis sentimen VTB disimpan di: ../../../stemming/outputs/VTB\sentiment_clean.csv
